# Challenge Status and Evidence Walkthrough

This notebook is a consolidated tour of what this repository implements for the Aqua challenge and runnable evidence for the implemented work. Run it from the repository root with the project's `.venv` kernel selected.

> **Scope honesty:** The repository implements Tasks 1–4. Bonus connection regrowth is not implemented and remains out of scope. The status table below links each implemented task to its source and tests.

In [ ]:
from pathlib import Path
import numpy as np

from engine import (
    Tensor, add, apply_mask, check_gradient, div, gelu, matmul, mean,
    mul, neg, power, relu, sigmoid, softmax, softmax_cross_entropy,
    sub, sum, tanh,
)
from nn import Adam, Linear, Module, Parameter, ReLU, Sequential, Sigmoid, Tanh
from prune import CubicPruningSchedule, TaylorPruner, taylor_saliency
from train.trainer import clip_grad_norm, train_classifier

np.set_printoptions(precision=4, suppress=True)
print("Repository root:", Path.cwd())
print("NumPy:", np.__version__)
print("All Task 1, 2, and 3 APIs imported successfully.")

## Implementation status

| Challenge area | Repository implementation | Evidence in this notebook / tests |
|---|---|---|
| Task 1 — reverse-mode autodiff | Tensor graph, arithmetic, matrix products, reductions, activations, cross-entropy, masking, finite-difference checker | Gradient checks and executable gradients; `tests/test_autodiff.py` |
| Task 2 — MLP and optimizer | NumPy `Linear`/activation/`Sequential` modules, AdamW, mini-batch training, clipping, metrics | End-to-end classifier learning curve; `tests/test_nn.py` and `tests/test_training.py` |
| Task 3 — gradual pruning | Taylor saliency, global monotonic pruning, cubic schedule, training-loop integration, Adam state clearing | Integrated scheduled-pruning run; `tests/test_pruning.py` |
| Bonus — connection regrowth | Not implemented | Explicitly out of scope; pruned masks are monotonic |
| Part 4 — Pareto sweeps | Taylor/magnitude sparsity sweep, held-out synthetic accuracy, FLOPs, CSR inference, raw metrics, and plot | `train/sweep.py`, `tests/test_sweep.py`, and `results/` |

Part 4 uses a deterministic synthetic classification dataset because the project does not include MNIST. Its reported accuracy is not an MNIST or production claim.

## Evidence 1: Task 1 gradient correctness

The finite-difference checker compares reverse-mode gradients to numerical central differences. This example combines broadcasting, multiplication, power, and a reduction into a scalar objective.

In [ ]:
def objective(x):
    return mean((x * x + 2.0 * x - 1.0) ** 2)

gradient_error = check_gradient(objective, np.array([-0.8, 0.2, 1.1]))
x = Tensor([-0.8, 0.2, 1.1], requires_grad=True)
objective_value = objective(x)
objective_value.backward()
print("objective:", objective_value.data.item())
print("reverse-mode gradient:", x.grad)
print("finite-difference relative error:", gradient_error)
assert gradient_error < 1e-5
assert np.isfinite(x.grad).all()

The full Task 1 exported API is present. The examples below show representative operator/function pairs, reductions, activations, stable classification loss, and masked-weight gradients; deeper hands-on examples are in `notebooks/autodiff_playground.ipynb`.

In [ ]:
left = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
bias = Tensor([0.5, -0.5], requires_grad=True)
print("add/operator:", (left + bias).data)
print("sub/function:", sub(left, bias).data)
print("mul/function:", mul(left, 2.0).data)
print("div/function:", div(left, 2.0).data)
print("neg:", neg(left).data)
print("power:", power(left, 2).data)
print("matmul:", matmul(left, np.eye(2)).data)
print("sum(axis=0):", sum(left, axis=0).data)
print("mean:", mean(left).data)

activation_input = Tensor([-1.0, 0.0, 1.0])
print("relu/tanh/sigmoid/gelu:",
      relu(activation_input).data,
      tanh(activation_input).data,
      sigmoid(activation_input).data,
      gelu(activation_input).data)

logits = Tensor([[2.0, 0.0, -1.0], [0.0, 1.0, 2.0]], requires_grad=True)
targets = np.array([[1.0, 0.0, 0.0], [0.0, 0.0, 1.0]])
loss = softmax_cross_entropy(logits, targets)
loss.backward()
print("softmax probabilities:\n", softmax(logits).data)
print("cross-entropy:", loss.data.item(), "logit grad finite:", np.isfinite(logits.grad).all())

masked_source = Tensor([[2.0, 5.0]], requires_grad=True)
masked = apply_mask(masked_source, np.array([[1.0, 0.0]]))
sum(masked).backward()
print("masked value:", masked.data, "gradient:", masked_source.grad)
np.testing.assert_array_equal(masked.data, [[2.0, 0.0]])
np.testing.assert_array_equal(masked_source.grad, [[1.0, 0.0]])

## Evidence 2: Task 2 classification learning curve

Build a deterministic three-class synthetic dataset, compose a small MLP, and train it using the repository's AdamW and mini-batch training loop. The first Linear layer requests He initialization for ReLU; the logits layer uses Xavier initialization. We report epoch loss and accuracy and assert finite values and a lower final loss.

In [ ]:
data_rng = np.random.default_rng(22)
class_centers = np.array([[-2.0, -1.0], [2.0, -1.0], [0.0, 2.0]])
features = np.concatenate([
    center + data_rng.normal(0.0, 0.5, size=(60, 2))
    for center in class_centers
])
labels = np.repeat(np.arange(3), 60)
order = data_rng.permutation(len(labels))
features, labels = features[order], labels[order]

model_rng = np.random.default_rng(7)
model = Sequential(
    Linear(2, 12, activation="relu", rng=model_rng),
    ReLU(),
    Linear(12, 3, activation="linear", rng=model_rng),
)
optimizer = Adam(model.parameters(), lr=0.03, weight_decay=1e-4)
history = train_classifier(
    model, optimizer, features, labels,
    epochs=10, batch_size=32, seed=11, max_grad_norm=5.0,
)

print("epoch | loss     | accuracy")
for item in history:
    print(f"{item.epoch:5d} | {item.loss:8.5f} | {item.accuracy:8.3f}")
losses = np.array([item.loss for item in history])
accuracies = np.array([item.accuracy for item in history])
assert np.isfinite(losses).all() and np.isfinite(accuracies).all()
assert all(np.isfinite(p.data).all() for p in model.parameters())
assert losses[-1] < losses[0]
print("optimizer update count:", optimizer.t)
print("finite parameters and improving training loss: PASS")

## Evidence 3: Task 3 schedule, ranking, and integrated pruning

Taylor saliency is `abs(weight * gradient)`. The cubic schedule generates sparsity targets; the pruner applies those targets globally across all Linear weights at selected epoch boundaries. This run reports loss, accuracy, and measured sparsity at each epoch and checks exact final sparsity, finite metrics, monotonic masks, and zero values at all pruned coordinates.

In [ ]:
print("small Taylor example:", taylor_saliency([[2.0, -3.0]], [[0.1, 0.5]]))

prune_rng = np.random.default_rng(7)
pruned_model = Sequential(
    Linear(2, 8, activation="relu", rng=prune_rng),
    ReLU(),
    Linear(8, 3, activation="linear", rng=prune_rng),
)
schedule = CubicPruningSchedule(
    initial_sparsity=0.0,
    final_sparsity=0.5,
    begin_step=2,
    end_step=8,
    update_frequency=2,
)
pruning_optimizer = Adam(pruned_model.parameters(), lr=0.03, weight_decay=1e-4)
pruner = TaylorPruner(pruned_model)
pruned_history = train_classifier(
    pruned_model,
    pruning_optimizer,
    features,
    labels,
    epochs=8,
    batch_size=32,
    seed=11,
    pruning_schedule=schedule,
    pruner=pruner,
)

print("epoch | loss     | accuracy | sparsity")
for item in pruned_history:
    print(f"{item.epoch:5d} | {item.loss:8.5f} | {item.accuracy:8.3f} | {item.sparsity:8.3f}")

measured_sparsity = np.array([item.sparsity for item in pruned_history])
assert np.isfinite([item.loss for item in pruned_history]).all()
assert np.all(np.diff(measured_sparsity) >= 0)
assert np.isclose(pruner.sparsity, 0.5)
assert all(np.all(layer.W.data[layer.mask == 0] == 0.0) for layer in pruner.layers)
print("final pruner sparsity:", pruner.sparsity)
print("all masked weights exactly zero; sparsity is monotonic: PASS")

Pruning also clears Adam state for every removed weight. This notebook checks the moment and per-entry bias-correction buffers after the integrated run.

In [ ]:
for layer in pruner.layers:
    index = next(i for i, parameter in enumerate(pruning_optimizer.params) if parameter is layer.W)
    inactive = layer.mask == 0
    assert np.all(pruning_optimizer.m[index][inactive] == 0.0)
    assert np.all(pruning_optimizer.v[index][inactive] == 0.0)
    assert np.all(pruning_optimizer.step_counts[index][inactive] == 0)
print("All pruned weights have zero first moment, second moment, and Adam age.")
print("Per-connection regrowth is not implemented; current masks are monotonic.")

## Evidence 4: repository quality gate

The test suite is the executable regression gate across Tasks 1–4. This cell runs `pytest -q tests` from the repository root. Bonus connection regrowth remains out of scope.

In [ ]:
import subprocess
import sys

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    cwd=Path.cwd(),
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"Repository test suite failed with exit code {completed.returncode}")
print("All repository tests passed.")

## Final assessment

The notebook demonstrates and tests the implemented path end to end: autodiff gradients -> MLP/AdamW training -> Taylor ranking -> cubic-scheduled global pruning -> mask and optimizer-state invariants, with Task 4 evaluation available in `train/sweep.py`.

**Scope:** Tasks 1–4 have runnable code and tests. Bonus connection regrowth is not implemented. The Part 4 results are the committed synthetic-data experiment, not a claim about MNIST or production.